<a href="https://colab.research.google.com/github/oojgu/museum-task/blob/main/%D1%83%D1%87%D0%B5%D1%82_%D0%BF%D1%80%D0%BE%D0%B5%D0%BA%D1%82%D0%BE%D0%B2_%D0%B8_%D0%B7%D0%B0%D0%B4%D0%B0%D1%87.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import sys
import psycopg2
import pandas as pd

print("ЗАПУСК ПРОЕКТА: Учет заявок на ремонт (Вариант 1)")

if 'google.colab' in sys.modules:
    print("Установка PostgreSQL в Colab...")
    !sudo apt-get update -qq > /dev/null 2>&1
    !sudo apt-get install postgresql postgresql-contrib -qq > /dev/null 2>&1
    !sudo service postgresql start
    !sudo -u postgres psql -c "ALTER USER postgres WITH PASSWORD 'postgres';" > /dev/null 2>&1
    !sudo -u postgres psql -c "CREATE DATABASE repair_db;" > /dev/null 2>&1
    print("PostgreSQL установлен и запущен!")
else:
    print("Вы не в Colab, используйте существующий PostgreSQL")

print("\nПодключение к базе данных...")

conn = psycopg2.connect(
    host="localhost",
    database="repair_db",
    user="postgres",
    password="postgres"
)
cursor = conn.cursor()
print("Подключение установлено!")

print("\nБЛОК 3: СОЗДАНИЕ ТАБЛИЦ")

cursor.execute("CREATE EXTENSION IF NOT EXISTS pgcrypto;")
print("Расширение pgcrypto включено")

cursor.execute("""
DROP TABLE IF EXISTS repair_requests CASCADE;
DROP TABLE IF EXISTS equipment CASCADE;
DROP TABLE IF EXISTS users CASCADE;
""")
print("Старые таблицы удалены")

cursor.execute("""
CREATE TABLE users (
    id SERIAL PRIMARY KEY,
    username TEXT NOT NULL UNIQUE,
    email TEXT NOT NULL UNIQUE,
    password_hash TEXT NOT NULL,
    role TEXT NOT NULL CHECK (role IN ('employee', 'master')),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
""")
print("Таблица 'users' создана")

cursor.execute("""
CREATE TABLE equipment (
    id SERIAL PRIMARY KEY,
    name TEXT NOT NULL,
    inventory_number TEXT NOT NULL UNIQUE
);
""")
print("Таблица 'equipment' создана")

cursor.execute("""
CREATE TABLE repair_requests (
    id SERIAL PRIMARY KEY,
    equipment_id INTEGER NOT NULL REFERENCES equipment(id) ON DELETE RESTRICT,
    created_by INTEGER NOT NULL REFERENCES users(id) ON DELETE RESTRICT,
    assigned_to INTEGER REFERENCES users(id) ON DELETE SET NULL,
    description TEXT NOT NULL,
    status TEXT NOT NULL DEFAULT 'new'
        CHECK (status IN ('new', 'in_progress', 'completed', 'cancelled')),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
""")
print("Таблица 'repair_requests' создана")

cursor.execute("CREATE INDEX idx_repair_requests_equipment ON repair_requests(equipment_id);")
cursor.execute("CREATE INDEX idx_repair_requests_created_by ON repair_requests(created_by);")
cursor.execute("CREATE INDEX idx_repair_requests_assigned_to ON repair_requests(assigned_to);")
cursor.execute("CREATE INDEX idx_repair_requests_status ON repair_requests(status);")
cursor.execute("CREATE INDEX idx_equipment_inventory ON equipment(inventory_number);")
print("Индексы созданы")

conn.commit()

print("\nБЛОК 4: ЗАПОЛНЕНИЕ ТАБЛИЦ ДАННЫМИ")

cursor.execute("""
INSERT INTO users (username, email, password_hash, role) VALUES
    ('Анна_С', 'anna.s@company.ru', crypt('AnnaPass123', gen_salt('bf')), 'employee'),
    ('Пётр_М', 'petr.m@company.ru', crypt('PetrMaster456', gen_salt('bf')), 'master'),
    ('Мария_С', 'maria.i@company.ru', crypt('Maria789', gen_salt('bf')), 'employee'),
    ('Иван_М', 'ivan.k@company.ru', crypt('IvanMaster777', gen_salt('bf')), 'master'),
    ('Елена_С', 'elena.p@company.ru', crypt('Elena321', gen_salt('bf')), 'employee');
""")
print("Добавлено 5 пользователей")

cursor.execute("""
INSERT INTO equipment (name, inventory_number) VALUES
    ('Принтер HP LaserJet Pro', 'INV-001'),
    ('Ноутбук Lenovo ThinkPad', 'INV-002'),
    ('МФУ Kyocera ECOSYS', 'INV-003'),
    ('Сервер DELL PowerEdge', 'INV-004'),
    ('Монитор Samsung 24"', 'INV-005'),
    ('Климатическая установка', 'INV-006');
""")
print("Добавлено 6 единиц оборудования")

cursor.execute("""
INSERT INTO repair_requests (equipment_id, created_by, assigned_to, description, status, created_at) VALUES
    (1, 1, 2, 'Не включается принтер', 'completed', '2025-05-10 09:30:00'),
    (2, 3, 4, 'Черный экран при загрузке', 'in_progress', '2025-06-01 14:20:00'),
    (3, 1, 2, 'Застревает бумага', 'completed', '2025-05-15 11:00:00'),
    (4, 5, 4, 'Периодически отключается сервер', 'new', '2025-06-03 10:00:00'),
    (5, 3, 2, 'Мерцает экран', 'cancelled', '2025-05-20 08:00:00'),
    (6, 1, 4, 'Не охлаждает', 'new', '2025-06-04 16:45:00'),
    (1, 3, 2, 'Принтер не видит картридж', 'completed', '2025-05-12 12:30:00'),
    (2, 5, 4, 'Сломался USB-порт', 'completed', '2025-05-05 09:00:00'),
    (3, 1, 2, 'Не сканирует', 'in_progress', '2025-05-28 10:15:00'),
    (4, 3, 4, 'Шум вентиляторов', 'cancelled', '2025-05-25 13:30:00'),
    (2, 1, 4, 'Зависает при открытии', 'in_progress', '2025-06-02 11:00:00'),
    (5, 5, 2, 'Пропадает изображение', 'new', '2025-06-04 09:00:00'),
    (3, 3, 2, 'Шумит при работе', 'completed', '2025-05-18 14:00:00'),
    (1, 5, 2, 'Печатает полосами', 'completed', '2025-05-22 10:30:00'),
    (6, 3, 4, 'Вода не отводится', 'new', '2025-06-05 08:00:00'),
    (4, 1, 4, 'Сервер перегревается', 'in_progress', '2025-06-01 15:00:00'),
    (2, 5, 4, 'Не работает Wi-Fi', 'new', '2025-06-03 12:00:00'),
    (5, 1, 2, 'Кнопки не реагируют', 'completed', '2025-05-25 16:00:00'),
    (3, 5, 2, 'Ошибка на экране', 'in_progress', '2025-05-30 09:00:00'),
    (1, 3, 2, 'Нет питания', 'completed', '2025-05-28 11:30:00'),
    (6, 1, 4, 'Странный запах', 'cancelled', '2025-05-20 10:00:00'),
    (2, 3, 4, 'Тормозит система', 'in_progress', '2025-06-04 14:00:00');
""")
print("Добавлено 22 заявки на ремонт")

conn.commit()

print("\nБЛОК 5: БЕЗОПАСНАЯ АУТЕНТИФИКАЦИЯ")

def authenticate(username, password):
    cursor.execute("""
        SELECT id, username, email, role, created_at FROM users
        WHERE username = %s AND password_hash = crypt(%s, password_hash)
    """, (username, password))
    return cursor.fetchone()

print("\nТестирование аутентификации:")

user = authenticate("Анна_С", "AnnaPass123")
if user:
    print(f"Успешный вход: {user[1]} (ID: {user[0]}, Роль: {user[3]})")
else:
    print("Ошибка входа")

user = authenticate("Анна_С", "wrong_password")
if user:
    print("Неверный пароль, но вход выполнен! (ОШИБКА)")
else:
    print("Неверный пароль - доступ запрещен")

print("\nБЛОК 6: ЗАЩИТА ОТ SQL-ИНЪЕКЦИЙ")

print("\nУЯЗВИМЫЙ ПОДХОД (антипаттерн):")
print("""
   username = "Анна_С' --"
   query = f"SELECT * FROM users WHERE username = '{username}'"
   Результат: SELECT * FROM users WHERE username = 'Анна_С' --'
   Комментарий -- отключает проверку пароля!
""")

print("\nБЕЗОПАСНЫЙ ПОДХОД (параметризация):")
print("""
   username = "Анна_С' --"
   cursor.execute("SELECT * FROM users WHERE username = %s", (username,))
   Данные интерпретируются как строка, а не как SQL-код
""")

print("\nТЕСТ: Попытка SQL-инъекции через нашу защищенную функцию")
malicious_input = "Анна_С' --"
result = authenticate(malicious_input, "any_password")
if result:
    print(f"АТАКА УСПЕШНА! Найден пользователь: {result[1]}")
else:
    print(f"АТАКА ОТБИТА! Пользователь '{malicious_input}' не найден")

print("\nБЛОК 7: ЗАПРОСЫ ДЛЯ ДАШБОРДА")

print("\nЗапрос 1: Статистика заявок по статусам")
cursor.execute("""
    SELECT
        status,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS percentage
    FROM repair_requests
    GROUP BY status
    ORDER BY
        CASE status
            WHEN 'new' THEN 1
            WHEN 'in_progress' THEN 2
            WHEN 'completed' THEN 3
            WHEN 'cancelled' THEN 4
        END;
""")
rows = cursor.fetchall()
print(f"\n   {'Статус':<15} {'Количество':<12} {'Процент':<10}")
print("   " + "-" * 40)
for row in rows:
    print(f"   {row[0]:<15} {row[1]:<12} {row[2]:<10}%")

print("\nЗапрос 2: Топ-5 оборудования, которое чаще всего ломается")
cursor.execute("""
    SELECT
        e.name,
        e.inventory_number,
        COUNT(rr.id) AS repair_count
    FROM equipment e
    LEFT JOIN repair_requests rr ON e.id = rr.equipment_id
    GROUP BY e.id, e.name, e.inventory_number
    ORDER BY repair_count DESC
    LIMIT 5;
""")
rows = cursor.fetchall()
print(f"\n   {'Оборудование':<30} {'Инв. номер':<15} {'Кол-во поломок':<15}")
print("   " + "-" * 60)
for row in rows:
    print(f"   {row[0][:28]:<30} {row[1]:<15} {row[2]:<15}")

print("\nЗапрос 3: Загрузка мастеров")
cursor.execute("""
    SELECT
        u.username,
        COUNT(CASE WHEN rr.status IN ('new', 'in_progress') THEN 1 END) AS active_requests,
        COUNT(CASE WHEN rr.status = 'completed' THEN 1 END) AS completed_requests,
        COUNT(rr.id) AS total_assigned
    FROM users u
    LEFT JOIN repair_requests rr ON u.id = rr.assigned_to
    WHERE u.role = 'master'
    GROUP BY u.id, u.username
    ORDER BY active_requests DESC;
""")
rows = cursor.fetchall()
print(f"\n   {'Мастер':<15} {'Активных':<12} {'Завершено':<12} {'Всего':<10}")
print("   " + "-" * 50)
for row in rows:
    print(f"   {row[0]:<15} {row[1]:<12} {row[2]:<12} {row[3]:<10}")

print("\nЗАВЕРШЕНИЕ РАБОТЫ")

cursor.close()
conn.close()
print("Соединение с базой данных закрыто")

ЗАПУСК ПРОЕКТА: Учет заявок на ремонт (Вариант 1)
Установка PostgreSQL в Colab...
 * Starting PostgreSQL 14 database server
   ...done.
PostgreSQL установлен и запущен!

Подключение к базе данных...
Подключение установлено!

БЛОК 3: СОЗДАНИЕ ТАБЛИЦ
Расширение pgcrypto включено
Старые таблицы удалены
Таблица 'users' создана
Таблица 'equipment' создана
Таблица 'repair_requests' создана
Индексы созданы

БЛОК 4: ЗАПОЛНЕНИЕ ТАБЛИЦ ДАННЫМИ
Добавлено 5 пользователей
Добавлено 6 единиц оборудования
Добавлено 22 заявки на ремонт

БЛОК 5: БЕЗОПАСНАЯ АУТЕНТИФИКАЦИЯ

Тестирование аутентификации:
Успешный вход: Анна_С (ID: 1, Роль: employee)
Неверный пароль - доступ запрещен

БЛОК 6: ЗАЩИТА ОТ SQL-ИНЪЕКЦИЙ

УЯЗВИМЫЙ ПОДХОД (антипаттерн):

   username = "Анна_С' --"
   query = f"SELECT * FROM users WHERE username = '{username}'"
   Результат: SELECT * FROM users WHERE username = 'Анна_С' --'
   Комментарий -- отключает проверку пароля!


БЕЗОПАСНЫЙ ПОДХОД (параметризация):

   username = "Анна_С' --